# MODULE 2 — Feature Engineering Spatial
## Cours Magistral Interactif

> **Cours** : Analyse Spatiale avec Machine Learning (ASML)  
> **Institut** : 2iE — Master Eau, Environnement, Aménagement  
> **Version** : 3 — Données réelles GADM 4.1 · Cadre Harmonisé IPC · FAPAR Copernicus

---

## Comment utiliser ce notebook

Ce notebook est la **version interactive** du support de cours CM (document Word).
Il vous permet de **lire la théorie et exécuter les exemples** dans le même environnement.

| Type de cellule | Contenu |
|----------------|---------|
| Cellules Markdown | Théorie, définitions, explications, callouts pédagogiques |
| Cellules Code | Exemples illustratifs **exécutables** — pas des exercices |

> ⚠️ Les **exercices** sont dans `M2_TP_Enonce_v3.ipynb`.
> Ce notebook sert à comprendre les concepts avant de les appliquer.

---

## Question centrale du module

> *Quelles variables géospatiales permettent de prédire si une province du Burkina Faso*
> *sera en phase alimentaire 1 (Minimal), 3 (Crise) ou 4 (Urgence) ?*

| Section | Feature produite | Question répondue |
|---------|-----------------|------------------|
| §3 | superficie_km2, compacite, pop_density | Quelle est la forme et la charge humaine ? |
| §4 | ipc_lag, lisa_cat, Moran I | Les voisines sont-elles aussi en crise ? |
| §5 | dist_route_km, travel_time_h | La province est-elle enclavée ? |
| §6 | fapar_mean, fapar_trend | La végétation suffit-elle pour une récolte ? |
| §7 | Feature matrix VIF-validée | Quelles features retenir pour le ML ? |

## Installation (Google Colab uniquement)

In [ ]:
# Décommenter si nécessaire (Google Colab)
# !pip install geopandas libpysal esda rasterstats statsmodels folium
#             contextily requests scipy scikit-learn --quiet

## Imports

In [ ]:
import json, io, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests
from shapely.geometry import Point, LineString
from sklearn.preprocessing import MinMaxScaler
from scipy.spatial import cKDTree
import folium

from libpysal.weights import Queen, KNN
from libpysal.weights import lag_spatial
from esda.moran import Moran, Moran_Local
from statsmodels.stats.outliers_influence import variance_inflation_factor

TIMEOUT = 15
print('✅ Bibliothèques chargées. GeoPandas :', gpd.__version__)

# ═══════════════════════════════════════════════════════════════
# Données fallback embarquées — définies EN PREMIER
# (avant load_gadm pour éviter tout NameError)
# 20 provinces représentatives du gradient sahélien BF
# ═══════════════════════════════════════════════════════════════

FALLBACK_GJ_STR = '{"type": "FeatureCollection", "features": [{"type": "Feature", "properties": {"province": "Oudalan", "region": "Sahel", "pop": 322000, "pluvio": 280, "ipc_phase": 4, "fapar_mean": 0.05}, "geometry": {"type": "Polygon", "coordinates": [[[-0.78, 14.23], [-0.18, 14.23], [-0.18, 14.73], [-0.78, 14.73], [-0.78, 14.23]]]}}, {"type": "Feature", "properties": {"province": "Séno", "region": "Sahel", "pop": 325000, "pluvio": 310, "ipc_phase": 4, "fapar_mean": 0.06}, "geometry": {"type": "Polygon", "coordinates": [[[-0.4, 13.85], [0.19999999999999998, 13.85], [0.19999999999999998, 14.35], [-0.4, 14.35], [-0.4, 13.85]]]}}, {"type": "Feature", "properties": {"province": "Soum", "region": "Sahel", "pop": 369000, "pluvio": 350, "ipc_phase": 4, "fapar_mean": 0.07}, "geometry": {"type": "Polygon", "coordinates": [[[-1.85, 13.73], [-1.25, 13.73], [-1.25, 14.23], [-1.85, 14.23], [-1.85, 13.73]]]}}, {"type": "Feature", "properties": {"province": "Yagha", "region": "Sahel", "pop": 214000, "pluvio": 420, "ipc_phase": 3, "fapar_mean": 0.09}, "geometry": {"type": "Polygon", "coordinates": [[[0.45, 13.1], [1.05, 13.1], [1.05, 13.6], [0.45, 13.6], [0.45, 13.1]]]}}, {"type": "Feature", "properties": {"province": "Loroum", "region": "Nord", "pop": 194000, "pluvio": 480, "ipc_phase": 3, "fapar_mean": 0.11}, "geometry": {"type": "Polygon", "coordinates": [[[-2.4, 13.45], [-1.8, 13.45], [-1.8, 13.95], [-2.4, 13.95], [-2.4, 13.45]]]}}, {"type": "Feature", "properties": {"province": "Yatenga", "region": "Nord", "pop": 535000, "pluvio": 520, "ipc_phase": 3, "fapar_mean": 0.13}, "geometry": {"type": "Polygon", "coordinates": [[[-2.65, 13.3], [-2.0500000000000003, 13.3], [-2.0500000000000003, 13.8], [-2.65, 13.8], [-2.65, 13.3]]]}}, {"type": "Feature", "properties": {"province": "Passoré", "region": "Nord", "pop": 313000, "pluvio": 560, "ipc_phase": 3, "fapar_mean": 0.15}, "geometry": {"type": "Polygon", "coordinates": [[[-2.5, 12.64], [-1.9000000000000001, 12.64], [-1.9000000000000001, 13.14], [-2.5, 13.14], [-2.5, 12.64]]]}}, {"type": "Feature", "properties": {"province": "Sanmatenga", "region": "Centre-Nord", "pop": 532000, "pluvio": 590, "ipc_phase": 3, "fapar_mean": 0.16}, "geometry": {"type": "Polygon", "coordinates": [[[-1.5, 12.8], [-0.8999999999999999, 12.8], [-0.8999999999999999, 13.3], [-1.5, 13.3], [-1.5, 12.8]]]}}, {"type": "Feature", "properties": {"province": "Namentenga", "region": "Centre-Nord", "pop": 331000, "pluvio": 620, "ipc_phase": 3, "fapar_mean": 0.18}, "geometry": {"type": "Polygon", "coordinates": [[[-0.8, 12.85], [-0.2, 12.85], [-0.2, 13.35], [-0.8, 13.35], [-0.8, 12.85]]]}}, {"type": "Feature", "properties": {"province": "Bam", "region": "Centre-Nord", "pop": 295000, "pluvio": 640, "ipc_phase": 2, "fapar_mean": 0.19}, "geometry": {"type": "Polygon", "coordinates": [[[-1.7, 13.2], [-1.0999999999999999, 13.2], [-1.0999999999999999, 13.7], [-1.7, 13.7], [-1.7, 13.2]]]}}, {"type": "Feature", "properties": {"province": "Kadiogo", "region": "Centre", "pop": 2415000, "pluvio": 820, "ipc_phase": 2, "fapar_mean": 0.25}, "geometry": {"type": "Polygon", "coordinates": [[[-1.83, 12.11], [-1.23, 12.11], [-1.23, 12.61], [-1.83, 12.61], [-1.83, 12.11]]]}}, {"type": "Feature", "properties": {"province": "Bazega", "region": "Centre-Sud", "pop": 226000, "pluvio": 850, "ipc_phase": 2, "fapar_mean": 0.28}, "geometry": {"type": "Polygon", "coordinates": [[[-1.7, 11.85], [-1.0999999999999999, 11.85], [-1.0999999999999999, 12.35], [-1.7, 12.35], [-1.7, 11.85]]]}}, {"type": "Feature", "properties": {"province": "Zoundweogo", "region": "Centre-Sud", "pop": 244000, "pluvio": 860, "ipc_phase": 2, "fapar_mean": 0.27}, "geometry": {"type": "Polygon", "coordinates": [[[-1.15, 11.6], [-0.55, 11.6], [-0.55, 12.1], [-1.15, 12.1], [-1.15, 11.6]]]}}, {"type": "Feature", "properties": {"province": "Nahouri", "region": "Centre-Sud", "pop": 185000, "pluvio": 870, "ipc_phase": 2, "fapar_mean": 0.26}, "geometry": {"type": "Polygon", "coordinates": [[[-1.4000000000000001, 11.25], [-0.8, 11.25], [-0.8, 11.75], [-1.4000000000000001, 11.75], [-1.4000000000000001, 11.25]]]}}, {"type": "Feature", "properties": {"province": "Boulgou", "region": "Centre-Est", "pop": 398000, "pluvio": 780, "ipc_phase": 3, "fapar_mean": 0.22}, "geometry": {"type": "Polygon", "coordinates": [[[-0.5, 11.6], [0.09999999999999998, 11.6], [0.09999999999999998, 12.1], [-0.5, 12.1], [-0.5, 11.6]]]}}, {"type": "Feature", "properties": {"province": "Ganzourgou", "region": "Plateau Central", "pop": 255000, "pluvio": 750, "ipc_phase": 2, "fapar_mean": 0.21}, "geometry": {"type": "Polygon", "coordinates": [[[-1.05, 12.1], [-0.45, 12.1], [-0.45, 12.6], [-1.05, 12.6], [-1.05, 12.1]]]}}, {"type": "Feature", "properties": {"province": "Houet", "region": "Hauts-Bassins", "pop": 1015000, "pluvio": 980, "ipc_phase": 2, "fapar_mean": 0.38}, "geometry": {"type": "Polygon", "coordinates": [[[-4.6, 10.93], [-4.0, 10.93], [-4.0, 11.43], [-4.6, 11.43], [-4.6, 10.93]]]}}, {"type": "Feature", "properties": {"province": "Comoé", "region": "Cascades", "pop": 274000, "pluvio": 1050, "ipc_phase": 1, "fapar_mean": 0.44}, "geometry": {"type": "Polygon", "coordinates": [[[-4.95, 10.35], [-4.3500000000000005, 10.35], [-4.3500000000000005, 10.85], [-4.95, 10.85], [-4.95, 10.35]]]}}, {"type": "Feature", "properties": {"province": "Poni", "region": "Sud-Ouest", "pop": 267000, "pluvio": 1100, "ipc_phase": 2, "fapar_mean": 0.48}, "geometry": {"type": "Polygon", "coordinates": [[[-3.5999999999999996, 10.05], [-3.0, 10.05], [-3.0, 10.55], [-3.5999999999999996, 10.55], [-3.5999999999999996, 10.05]]]}}, {"type": "Feature", "properties": {"province": "Noumbiel", "region": "Sud-Ouest", "pop": 100000, "pluvio": 1150, "ipc_phase": 2, "fapar_mean": 0.52}, "geometry": {"type": "Polygon", "coordinates": [[[-3.4, 9.75], [-2.8000000000000003, 9.75], [-2.8000000000000003, 10.25], [-3.4, 10.25], [-3.4, 9.75]]]}}]}'

fallback_gdf = gpd.GeoDataFrame.from_features(
    json.loads(FALLBACK_GJ_STR)['features'], crs='EPSG:4326'
)
print(f'Fallback : {len(fallback_gdf)} provinces embarquées')

# ═══════════════════════════════════════════════════════════════
# Chargement GADM 4.1 — provinces ou régions BF
# Source : https://gadm.org/download_country.html → GeoJSON
#   level-1 = 13 régions | level-2 = 45 provinces | level-3 = 351 communes
# Fallback automatique si GADM inaccessible
# ═══════════════════════════════════════════════════════════════

def load_gadm(level, fallback_gdf, label='données'):
    url = f'https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_BFA_{level}.json'
    try:
        r = requests.get(url, timeout=TIMEOUT); r.raise_for_status()
        gdf = gpd.read_file(io.BytesIO(r.content))
        col_map = {'NAME_0':'pays','NAME_1':'region','NAME_2':'province','NAME_3':'commune'}
        gdf = gdf.rename(columns={k:v for k,v in col_map.items() if k in gdf.columns})
        print(f'[GADM ✅] {len(gdf)} entités | CRS: {gdf.crs}')
        return gdf, 'GADM'
    except Exception as e:
        warnings.warn(f'GADM inaccessible ({e})', stacklevel=2)
        print(f'[FALLBACK ⚠️] {label} embarquées utilisées')
        return fallback_gdf.copy(), 'fallback'

# Données embarquées — 20 provinces représentatives du gradient sahélien BF
FALLBACK_GJ_STR = '{"type": "FeatureCollection", "features": [{"type": "Feature", "properties": {"province": "Oudalan", "region": "Sahel", "pop": 322000, "pluvio": 280, "ipc_phase": 4, "fapar_mean": 0.05}, "geometry": {"type": "Polygon", "coordinates": [[[-0.78, 14.23], [-0.18, 14.23], [-0.18, 14.73], [-0.78, 14.73], [-0.78, 14.23]]]}}, {"type": "Feature", "properties": {"province": "Séno", "region": "Sahel", "pop": 325000, "pluvio": 310, "ipc_phase": 4, "fapar_mean": 0.06}, "geometry": {"type": "Polygon", "coordinates": [[[-0.4, 13.85], [0.19999999999999998, 13.85], [0.19999999999999998, 14.35], [-0.4, 14.35], [-0.4, 13.85]]]}}, {"type": "Feature", "properties": {"province": "Soum", "region": "Sahel", "pop": 369000, "pluvio": 350, "ipc_phase": 4, "fapar_mean": 0.07}, "geometry": {"type": "Polygon", "coordinates": [[[-1.85, 13.73], [-1.25, 13.73], [-1.25, 14.23], [-1.85, 14.23], [-1.85, 13.73]]]}}, {"type": "Feature", "properties": {"province": "Yagha", "region": "Sahel", "pop": 214000, "pluvio": 420, "ipc_phase": 3, "fapar_mean": 0.09}, "geometry": {"type": "Polygon", "coordinates": [[[0.45, 13.1], [1.05, 13.1], [1.05, 13.6], [0.45, 13.6], [0.45, 13.1]]]}}, {"type": "Feature", "properties": {"province": "Loroum", "region": "Nord", "pop": 194000, "pluvio": 480, "ipc_phase": 3, "fapar_mean": 0.11}, "geometry": {"type": "Polygon", "coordinates": [[[-2.4, 13.45], [-1.8, 13.45], [-1.8, 13.95], [-2.4, 13.95], [-2.4, 13.45]]]}}, {"type": "Feature", "properties": {"province": "Yatenga", "region": "Nord", "pop": 535000, "pluvio": 520, "ipc_phase": 3, "fapar_mean": 0.13}, "geometry": {"type": "Polygon", "coordinates": [[[-2.65, 13.3], [-2.0500000000000003, 13.3], [-2.0500000000000003, 13.8], [-2.65, 13.8], [-2.65, 13.3]]]}}, {"type": "Feature", "properties": {"province": "Passoré", "region": "Nord", "pop": 313000, "pluvio": 560, "ipc_phase": 3, "fapar_mean": 0.15}, "geometry": {"type": "Polygon", "coordinates": [[[-2.5, 12.64], [-1.9000000000000001, 12.64], [-1.9000000000000001, 13.14], [-2.5, 13.14], [-2.5, 12.64]]]}}, {"type": "Feature", "properties": {"province": "Sanmatenga", "region": "Centre-Nord", "pop": 532000, "pluvio": 590, "ipc_phase": 3, "fapar_mean": 0.16}, "geometry": {"type": "Polygon", "coordinates": [[[-1.5, 12.8], [-0.8999999999999999, 12.8], [-0.8999999999999999, 13.3], [-1.5, 13.3], [-1.5, 12.8]]]}}, {"type": "Feature", "properties": {"province": "Namentenga", "region": "Centre-Nord", "pop": 331000, "pluvio": 620, "ipc_phase": 3, "fapar_mean": 0.18}, "geometry": {"type": "Polygon", "coordinates": [[[-0.8, 12.85], [-0.2, 12.85], [-0.2, 13.35], [-0.8, 13.35], [-0.8, 12.85]]]}}, {"type": "Feature", "properties": {"province": "Bam", "region": "Centre-Nord", "pop": 295000, "pluvio": 640, "ipc_phase": 2, "fapar_mean": 0.19}, "geometry": {"type": "Polygon", "coordinates": [[[-1.7, 13.2], [-1.0999999999999999, 13.2], [-1.0999999999999999, 13.7], [-1.7, 13.7], [-1.7, 13.2]]]}}, {"type": "Feature", "properties": {"province": "Kadiogo", "region": "Centre", "pop": 2415000, "pluvio": 820, "ipc_phase": 2, "fapar_mean": 0.25}, "geometry": {"type": "Polygon", "coordinates": [[[-1.83, 12.11], [-1.23, 12.11], [-1.23, 12.61], [-1.83, 12.61], [-1.83, 12.11]]]}}, {"type": "Feature", "properties": {"province": "Bazega", "region": "Centre-Sud", "pop": 226000, "pluvio": 850, "ipc_phase": 2, "fapar_mean": 0.28}, "geometry": {"type": "Polygon", "coordinates": [[[-1.7, 11.85], [-1.0999999999999999, 11.85], [-1.0999999999999999, 12.35], [-1.7, 12.35], [-1.7, 11.85]]]}}, {"type": "Feature", "properties": {"province": "Zoundweogo", "region": "Centre-Sud", "pop": 244000, "pluvio": 860, "ipc_phase": 2, "fapar_mean": 0.27}, "geometry": {"type": "Polygon", "coordinates": [[[-1.15, 11.6], [-0.55, 11.6], [-0.55, 12.1], [-1.15, 12.1], [-1.15, 11.6]]]}}, {"type": "Feature", "properties": {"province": "Nahouri", "region": "Centre-Sud", "pop": 185000, "pluvio": 870, "ipc_phase": 2, "fapar_mean": 0.26}, "geometry": {"type": "Polygon", "coordinates": [[[-1.4000000000000001, 11.25], [-0.8, 11.25], [-0.8, 11.75], [-1.4000000000000001, 11.75], [-1.4000000000000001, 11.25]]]}}, {"type": "Feature", "properties": {"province": "Boulgou", "region": "Centre-Est", "pop": 398000, "pluvio": 780, "ipc_phase": 3, "fapar_mean": 0.22}, "geometry": {"type": "Polygon", "coordinates": [[[-0.5, 11.6], [0.09999999999999998, 11.6], [0.09999999999999998, 12.1], [-0.5, 12.1], [-0.5, 11.6]]]}}, {"type": "Feature", "properties": {"province": "Ganzourgou", "region": "Plateau Central", "pop": 255000, "pluvio": 750, "ipc_phase": 2, "fapar_mean": 0.21}, "geometry": {"type": "Polygon", "coordinates": [[[-1.05, 12.1], [-0.45, 12.1], [-0.45, 12.6], [-1.05, 12.6], [-1.05, 12.1]]]}}, {"type": "Feature", "properties": {"province": "Houet", "region": "Hauts-Bassins", "pop": 1015000, "pluvio": 980, "ipc_phase": 2, "fapar_mean": 0.38}, "geometry": {"type": "Polygon", "coordinates": [[[-4.6, 10.93], [-4.0, 10.93], [-4.0, 11.43], [-4.6, 11.43], [-4.6, 10.93]]]}}, {"type": "Feature", "properties": {"province": "Comoé", "region": "Cascades", "pop": 274000, "pluvio": 1050, "ipc_phase": 1, "fapar_mean": 0.44}, "geometry": {"type": "Polygon", "coordinates": [[[-4.95, 10.35], [-4.3500000000000005, 10.35], [-4.3500000000000005, 10.85], [-4.95, 10.85], [-4.95, 10.35]]]}}, {"type": "Feature", "properties": {"province": "Poni", "region": "Sud-Ouest", "pop": 267000, "pluvio": 1100, "ipc_phase": 2, "fapar_mean": 0.48}, "geometry": {"type": "Polygon", "coordinates": [[[-3.5999999999999996, 10.05], [-3.0, 10.05], [-3.0, 10.55], [-3.5999999999999996, 10.55], [-3.5999999999999996, 10.05]]]}}, {"type": "Feature", "properties": {"province": "Noumbiel", "region": "Sud-Ouest", "pop": 100000, "pluvio": 1150, "ipc_phase": 2, "fapar_mean": 0.52}, "geometry": {"type": "Polygon", "coordinates": [[[-3.4, 9.75], [-2.8000000000000003, 9.75], [-2.8000000000000003, 10.25], [-3.4, 10.25], [-3.4, 9.75]]]}}]}'
fallback_gdf = gpd.GeoDataFrame.from_features(
    json.loads(FALLBACK_GJ_STR)['features'], crs='EPSG:4326'
)

# Chargement principal
gdf, src = load_gadm(2, fallback_gdf, '45 provinces BF')
print(f'Source : {src} | {len(gdf)} provinces')

## Chargement des données — pattern standard ASML

> Ce bloc est identique dans tous les notebooks du cours.
> Il essaie GADM 4.1 (données réelles), bascule automatiquement sur les données
> embarquées si la connexion échoue. Un message indique toujours la source utilisée.

---
# §0 — Fil Conducteur : de la géographie à la prédiction ML

## Le Cadre Harmonisé IPC — comprendre la variable cible

Le **Cadre Harmonisé (CH)** est le système de référence pour l'analyse de
l'insécurité alimentaire en Afrique de l'Ouest. Il classe chaque province
en **5 phases IPC** basées sur des indicateurs de terrain.

| Phase | Niveau | Signification terrain |
|-------|--------|----------------------|
| **1** | Minimal | Ménages couvrent leurs besoins normalement |
| **2** | Stress | Alimentation suffisante mais stratégies d'adaptation requises |
| **3** | **Crise** | Déficits alimentaires, dégradation des moyens de subsistance |
| **4** | **Urgence** | Déficits sévères, émaciation élevée |
| **5** | Famine | Mortalité ou émaciation extrêmes |

> 🎯 **ipc_phase est la variable y du modèle ML.**
> Toutes les features construites dans ce module sont les variables X.

## Pourquoi le feature engineering est nécessaire

Un algorithme ML ne peut traiter que des **tableaux numériques**.
Il ne comprend pas un shapefile, une image satellite ou un réseau routier.
Le feature engineering consiste à **traduire ces objets géographiques en nombres**.

In [ ]:
# Ajout des phases IPC au GeoDataFrame
# Simulation basée sur le gradient latitudinal (proxy du gradient climatique BF)
np.random.seed(42)
lat = gdf.geometry.centroid.y.values
phases = np.clip(
    np.round(1 + (lat - 9.5)/1.1 + np.random.normal(0, 0.4, len(gdf))).astype(int),
    1, 5)
gdf['ipc_phase'] = phases
gdf['ipc_label'] = [{1:'Minimal',2:'Stress',3:'Crise',4:'Urgence',5:'Famine'}[p]
                     for p in phases]

# Distribution des phases
print('Distribution des phases IPC :')
print(gdf['ipc_phase'].value_counts().sort_index())

# Visualisation rapide
fig, ax = plt.subplots(figsize=(8,4))
palette = {1:'#27ae60',2:'#f1c40f',3:'#e67e22',4:'#e74c3c',5:'#8e44ad'}
counts = gdf['ipc_phase'].value_counts().sort_index()
lbls = {1:'Minimal',2:'Stress',3:'Crise',4:'Urgence',5:'Famine'}
ax.bar([lbls[i] for i in counts.index], counts.values,
       color=[palette[i] for i in counts.index], edgecolor='white')
ax.set_title('Distribution des phases IPC — Burkina Faso', fontweight='bold')
ax.set_ylabel('Nombre de provinces')
plt.tight_layout(); plt.show()

---
# §1 — Pourquoi du Feature Engineering en Contexte Spatial ?

## 1.1 La dépendance spatiale — problème ET ressource

**Première loi de Tobler (1970)** : *"Tout est lié à tout, mais les choses proches
le sont plus que les choses lointaines."*

Cette loi a deux conséquences pour notre modèle ML :

| Conséquence | Nature | Impact |
|------------|--------|--------|
| Les observations ne sont PAS indépendantes | 🔴 Problème | K-fold CV classique surestime les performances |
| Les valeurs des voisins prédisent la valeur locale | 🟢 Ressource | Le lag spatial est une feature puissante |

## 1.2 Démonstration : la dépendance spatiale existe-t-elle dans nos données ?

> Avant de construire des features de voisinage, vérifions que la dépendance
> spatiale est réelle dans nos données IPC — via le diagramme de dispersion.

In [ ]:
# Démonstration visuelle de la dépendance spatiale
# On montre que les provinces proches ont tendance à avoir des IPC similaires

gdf_utm = gdf.to_crs(32630)
coords = np.array([(g.centroid.x, g.centroid.y) for g in gdf_utm.geometry])

# Calculer les distances inter-provinces
from scipy.spatial.distance import pdist, squareform
dist_matrix = squareform(pdist(coords)) / 1e3  # en km

# Calculer les différences IPC entre toutes les paires
ipc = gdf['ipc_phase'].values
diff_ipc = np.abs(ipc[:, None] - ipc[None, :])  # matrice différences

# Sélectionner uniquement les paires (triangle supérieur)
mask = np.triu(np.ones_like(dist_matrix, dtype=bool), k=1)
dists_pairs = dist_matrix[mask]
diffs_pairs = diff_ipc[mask]

# Scatter plot : distance vs différence IPC
fig, ax = plt.subplots(figsize=(9,5))

# Moyenner par tranches de distance
bins = np.arange(0, dists_pairs.max()+50, 50)
bin_idx = np.digitize(dists_pairs, bins)
bin_means = [diffs_pairs[bin_idx==i].mean() if (bin_idx==i).any() else np.nan
             for i in range(1, len(bins))]
bin_centers = bins[:-1] + 25

ax.scatter(dists_pairs, diffs_pairs, alpha=0.15, s=10, color='steelblue')
ax.plot(bin_centers, bin_means, 'r-o', linewidth=2, markersize=6,
        label='Différence IPC moyenne par tranche de 50 km')
ax.set_xlabel('Distance entre provinces (km)', fontsize=11)
ax.set_ylabel('Différence de phase IPC |iᵢ - iⱼ|', fontsize=11)
ax.set_title('Dépendance spatiale des phases IPC\nProvinces proches → IPC plus similaires', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print('Observation : la différence IPC AUGMENTE avec la distance.')
print('→ Les provinces proches ont des phases IPC plus similaires.')
print('→ Loi de Tobler vérifiée sur nos données.')

---
# §2 — Préparation et Harmonisation des Données

## 2.1 La règle absolue des CRS

> ⚠️ **Toutes les opérations métriques (superficie, distance, buffer) nécessitent
> un CRS en mètres (EPSG:32630).** Calculer en degrés (EPSG:4326) donne des
> résultats physiquement absurdes, sans aucun message d'erreur.

| CRS | EPSG | Usage correct |
|-----|------|---------------|
| WGS84 Géographique | 4326 | Stockage, échange, Folium |
| UTM Zone 30N | **32630** | **Tout calcul métrique BF** |
| Web Mercator | 3857 | Fonds de carte Contextily |

## 2.2 Démonstration — l'erreur silencieuse de la superficie en degrés

In [ ]:
# DÉMONSTRATION : erreur silencieuse si on calcule en EPSG:4326
print('=== EPSG:4326 (FAUX) ===')
area_deg2 = gdf.geometry.area.mean()
print(f'Superficie moyenne : {area_deg2:.4f} degrés² — sans signification physique !')

# Correction : reprojeter d'abord
print('\n=== EPSG:32630 (CORRECT) ===')
gdf_utm = gdf.to_crs(epsg=32630)
area_km2 = (gdf_utm.geometry.area / 1e6).mean()
print(f'Superficie moyenne : {area_km2:,.0f} km² — valeur cohérente pour le BF')
print(f'(Le BF fait ~274 000 km² / 45 provinces ≈ {274000/45:,.0f} km² en moyenne)')

# Règle mnémotechnique
print('\n💡 Règle : Je mesure → 32630 | Je stocke/affiche → 4326 | Fond carte → 3857')

## 2.3 Nettoyage géométrique — pourquoi c'est important

Des géométries invalides (auto-intersections, polygones dégénérés) produisent
des valeurs NaN ou aberrantes lors des calculs de distances et de zonal_stats.
Le nettoyage préventif est une bonne pratique systématique.

In [ ]:
# Vérification et correction des géométries
invalid = gdf[~gdf.is_valid]
print(f'Géométries invalides : {len(invalid)}')

if len(invalid) > 0:
    print('Correction avec buffer(0)...')
    gdf.geometry = gdf.geometry.buffer(0)
    print(f'Après correction : {(~gdf.is_valid).sum()} invalides restantes')

# Supprimer les nulles
gdf = gdf.dropna(subset=['geometry'])
print(f'GeoDataFrame propre : {len(gdf)} entités')
print(f'Types géométriques : {gdf.geometry.geom_type.value_counts().to_dict()}')

---
# §3 — Features Géométriques et de Position

## 3.1 Superficie, périmètre et compacité

> 🎯 **Pourquoi ces features prédisent-elles l'insécurité alimentaire ?**
> Une grande province est difficile à couvrir logistiquement lors d'une crise humanitaire.
> Une province peu compacte (allongée, fragmentée) rend la distribution de l'aide encore plus complexe.

**L'indice de compacité de Polsby-Popper** mesure la circularité d'une forme :

$$\text{compacité} = \frac{4\pi \times \text{aire}}{\text{périmètre}^2}$$

- Compacité = 1 → cercle parfait (province très accessible)
- Compacité → 0 → forme très allongée ou fragmentée (difficile à couvrir)

In [ ]:
# Calcul des features géométriques
gdf_utm = gdf.to_crs(epsg=32630)

gdf['superficie_km2'] = gdf_utm.geometry.area   / 1e6
gdf['perimetre_km']   = gdf_utm.geometry.length / 1e3
gdf['compacite']      = (4 * np.pi * gdf_utm.geometry.area
                          / gdf_utm.geometry.length**2)
gdf['centroid_lon']   = gdf.geometry.centroid.x
gdf['centroid_lat']   = gdf.geometry.centroid.y

# Aperçu des résultats
col = 'province' if 'province' in gdf.columns else gdf.columns[0]
print('=== Features géométriques (5 exemples) ===')
print(gdf[[col,'superficie_km2','perimetre_km','compacite']]
        .sort_values('superficie_km2', ascending=False).head(5).to_string(index=False))

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

gdf.plot(column='superficie_km2', cmap='Blues', legend=True,
         legend_kwds={'label':'Superficie (km²)','orientation':'vertical'},
         edgecolor='white', linewidth=0.5, ax=axes[0])
axes[0].set_title('Superficie des provinces BF', fontweight='bold')
axes[0].axis('off')

gdf.plot(column='compacite', cmap='RdYlGn', legend=True, vmin=0.2, vmax=0.9,
         legend_kwds={'label':'Compacité [0–1]','orientation':'vertical'},
         edgecolor='white', linewidth=0.5, ax=axes[1])
axes[1].set_title('Compacité (vert=accessible, rouge=allongée)', fontweight='bold')
axes[1].axis('off')

plt.suptitle('Features géométriques — Source : ' + src, fontsize=11)
plt.tight_layout(); plt.show()

## 3.2 Zone bioclimatique — encoder la connaissance experte

> 🌍 **Contexte BF** : le régime pluviométrique varie du simple au triple
> entre le Nord (< 400 mm/an) et le Sud (> 900 mm/an).
> La latitude est un proxy précis de ce gradient.
> Cette feature encode une connaissance experte que le modèle ne pourrait
> pas déduire seul à partir des données.

| Zone | Latitude | Pluvio | Système agricole |
|------|----------|--------|-----------------|
| Sahélien strict | > 14° | < 400 mm | Élevage pastoral |
| Sahélien | 12.5°–14° | 400–600 mm | Agriculture aléatoire |
| Soudano-sahélien | 11°–12.5° | 600–900 mm | Agriculture pluviale stable |
| Soudanien | < 11° | > 900 mm | Agriculture et élevage intégrés |

In [ ]:
def zone_bioclim(lat):
    if   lat > 14.0: return 'Sahélien strict'
    elif lat > 12.5: return 'Sahélien'
    elif lat > 11.0: return 'Soudano-sahélien'
    else:            return 'Soudanien'

gdf['zone_bioclim'] = gdf['centroid_lat'].apply(zone_bioclim)

# Lien zone bioclimatique - phase IPC
print('=== Lien zone bioclimatique → phase IPC moyenne ===')
print(gdf.groupby('zone_bioclim')['ipc_phase'].mean().sort_values(ascending=False))
print('\n→ Plus on va vers le Nord (aride), plus la phase IPC est élevée.')

## 3.3 Densité de population

> 🎯 **Pourquoi pop_density prédit-elle l'insécurité ?**
> Une province dense avec peu de terres agricoles fait face à une **pression accrue
> sur les ressources alimentaires**.
> À l'inverse, une province peu peuplée mais aride (Sahel) souffre d'une
> vulnérabilité **structurelle** différente : pas assez de production pour les habitants malgré leur faible nombre.

In [ ]:
# Densité de population
if 'pop' in gdf.columns:
    gdf['pop_density'] = gdf['pop'] / gdf['superficie_km2'].clip(lower=0.1)
else:
    # Simulation : pic centré sur Ouagadougou (-1.53, 12.36)
    np.random.seed(7)
    ouaga = np.exp(-((gdf['centroid_lon']+1.53)**2
                     +(gdf['centroid_lat']-12.36)**2)*0.3) * 600
    gdf['pop_density'] = ouaga + np.random.exponential(20, len(gdf))

col = 'province' if 'province' in gdf.columns else gdf.columns[0]
print('=== Pop. density — 5 plus élevées et 5 plus faibles ===')
sorted_df = gdf[[col,'pop_density','ipc_phase']].sort_values('pop_density', ascending=False)
print('Denses :', sorted_df.head(3).to_string(index=False))
print('Peu denses :', sorted_df.tail(3).to_string(index=False))

---
# §4 — Matrice de Poids et Autocorrélation Spatiale

## 4.1 La matrice W — encoder le voisinage

> 🎯 **Pourquoi construire W ?**
> Sans W, on ne peut pas calculer de lag spatial ni mesurer l'autocorrélation.
> W encode qui est voisin de qui — c'est le **fondement** de toute l'analyse spatiale.

**W[i,j] = 1** si les provinces i et j partagent une frontière (Queen),  
**W[i,j] = 0** sinon.  
Après normalisation en ligne : **Σⱼ W[i,j] = 1** pour chaque province i.

> ⚠️ Avec des rectangles fictifs (v1 du cours), deux provinces géographiquement
> éloignées mais dont les rectangles se touchent étaient considérées voisines.
> Les vraies géométries GADM éliminent ce biais.

In [ ]:
from libpysal.weights import Queen, KNN
from libpysal.weights import lag_spatial

# Construction de la matrice W Queen
ipc_vals = gdf['ipc_phase'].fillna(gdf['ipc_phase'].median()).values

W = Queen.from_dataframe(gdf, use_index=True)
W.transform = 'r'  # Normalisation en ligne — chaque ligne somme à 1

print(f'Matrice W construite :')
print(f'  Provinces     : {W.n}')
print(f'  Voisins moyen : {W.mean_neighbors:.1f}')
print(f'  Voisins min   : {W.min_neighbors}')
print(f'  Voisins max   : {W.max_neighbors}')

if W.islands:
    print(f'⚠️ {len(W.islands)} province(s) sans voisin → KNN k=4')
    W = KNN.from_dataframe(gdf, k=4); W.transform = 'r'
    print('W KNN construite en remplacement.')

## 4.2 Le lag spatial — transformer la dépendance en feature

Le lag spatial de la variable x pour la province i est :

$$\text{lag}_x[i] = \sum_j W[i,j] \cdot x[j]$$

C'est la **moyenne pondérée** de x dans le voisinage immédiat de i.

> 🎯 **ipc_lag ≠ ipc_phase** — ils mesurent des choses différentes :
> - `ipc_phase[i]` : situation actuelle de la province i
> - `ipc_lag[i]` : pression exercée par les provinces voisines sur i

> **Interprétation clé** : si `ipc_phase[i] = 2` et `ipc_lag[i] = 3.8`,
> la province i est stable mais entourée de provinces en crise →
> **signal précurseur** d'une détérioration probable.

In [ ]:
# Calcul du lag spatial
gdf['ipc_lag'] = lag_spatial(W, ipc_vals)

# Visualisation : IPC vs IPC_lag
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(gdf['ipc_phase'], gdf['ipc_lag'],
                     c=gdf['ipc_phase'], cmap='RdYlGn_r',
                     s=80, alpha=0.8, edgecolors='white', linewidth=0.5)
plt.colorbar(scatter, ax=ax, label='Phase IPC')
ax.plot([1,5],[1,5], 'k--', linewidth=1, alpha=0.5, label='ipc = ipc_lag (diagonale)')
ax.set_xlabel('Phase IPC locale (ipc_phase)', fontsize=11)
ax.set_ylabel('Phase IPC des voisines (ipc_lag)', fontsize=11)
ax.set_title('IPC local vs Lag spatial IPC\nLes points hors-diagonale = outliers spatiaux potentiels', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

# Annoter les outliers intéressants
col = 'province' if 'province' in gdf.columns else gdf.columns[0]
for _, row in gdf.iterrows():
    if abs(row['ipc_phase'] - row['ipc_lag']) > 0.8:
        ax.annotate(str(row[col])[:6],
                    xy=(row['ipc_phase'], row['ipc_lag']),
                    xytext=(4,4), textcoords='offset points', fontsize=7)

plt.tight_layout(); plt.show()

print('Provinces avec plus grande différence |ipc - ipc_lag| :')
gdf['diff_lag'] = abs(gdf['ipc_phase'] - gdf['ipc_lag'])
print(gdf[[col,'ipc_phase','ipc_lag','diff_lag']].nlargest(5,'diff_lag').to_string(index=False))

## 4.3 Indice de Moran I — mesurer l'autocorrélation globale

Le **Moran I** quantifie si les valeurs similaires ont tendance à se regrouper
géographiquement (clustering) ou à s'alterner (damier).

| Valeur de I | Interprétation | Implication ML |
|-------------|----------------|----------------|
| I ≈ 0, p > 0.05 | Distribution aléatoire | K-fold classique acceptable |
| **I > 0, p < 0.05** | **Clustering spatial** | **Spatial CV obligatoire** |
| I < 0, p < 0.05 | Damier spatial | Spatial CV recommandée |

> 🎯 **Deux utilités du Moran I dans ce module :**
> 1. Justifier la spatial cross-validation du Module 3 (si I > 0.3)
> 2. Valider que nos données IPC présentent bien une structure spatiale exploitable par le lag

In [ ]:
from esda.moran import Moran, Moran_Local

# Moran I global
moran = Moran(ipc_vals, W)

print('=== Test de Moran I — Phases IPC BF ===')
print(f'I = {moran.I:.4f}')
print(f'p = {moran.p_sim:.4f} (test par permutation, 999 simulations)')
print(f'Z = {moran.z_sim:.2f}')

if moran.p_sim < 0.05 and moran.I > 0:
    print()
    print('→ Clustering spatial SIGNIFICATIF des phases IPC.')
    print('→ Les provinces en crise sont regroupées géographiquement.')
    print('→ Spatial cross-validation OBLIGATOIRE en Module 3.')
    if moran.I > 0.3:
        print(f'→ I = {moran.I:.3f} > 0.3 : clustering fort — distance minimale entre folds > 100 km.')

## 4.4 LISA — localiser les hotspots et coldspots

Les indicateurs LISA (Local Indicators of Spatial Association) décomposent
le Moran global en contributions **locales**. Ils identifient :

| Quadrant | Condition | Signification terrain BF |
|----------|-----------|--------------------------|
| **HH** | ipc_i élevé + voisins élevés | Cluster de crise — hotspot humanitaire |
| **LL** | ipc_i faible + voisins faibles | Cluster de résilience — coldspot |
| **HL** | ipc_i élevé + voisins faibles | Province en crise isolée (choc local) |
| **LH** | ipc_i faible + voisins élevés | Province stable entourée de crises (à surveiller) |
| **NS** | p > 0.05 | Non significatif |

> 🤖 **lisa_cat devient une feature catégorielle** (encodée en one-hot).
> Elle donne au modèle le 'contexte régional' de chaque province.

In [ ]:
# LISA locaux
lisa = Moran_Local(ipc_vals, W, seed=42)
gdf['lisa_q']   = lisa.q
gdf['lisa_p']   = lisa.p_sim
gdf['lisa_cat'] = gdf.apply(
    lambda r: {1:'HH',2:'LH',3:'LL',4:'HL'}.get(int(r['lisa_q']),'NS')
              if r['lisa_p'] < 0.05 else 'NS', axis=1)

print('Répartition LISA :')
print(gdf['lisa_cat'].value_counts())

# Carte LISA
palette_lisa = {'HH':'#e74c3c','LL':'#2ecc71','HL':'#e67e22','LH':'#3498db','NS':'#bdc3c7'}
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

gdf.plot(color=gdf['lisa_cat'].map(palette_lisa),
         edgecolor='white', linewidth=0.5, ax=axes[0])
axes[0].set_title(f'Carte LISA — Phases IPC\nMoran I = {moran.I:.3f} (p = {moran.p_sim:.3f})',
                  fontweight='bold')
axes[0].axis('off')
leg = [mpatches.Patch(color=v,label={'HH':'HH — Hotspot','LL':'LL — Coldspot',
       'HL':'HL — Crise isolée','LH':'LH — Stable isolée','NS':'NS'}[k])
       for k,v in palette_lisa.items()]
axes[0].legend(handles=leg, loc='lower left', fontsize=8)

# Diagramme de Moran
ipc_z = (ipc_vals - ipc_vals.mean()) / ipc_vals.std()
lag_z  = lag_spatial(W, ipc_z)
axes[1].scatter(ipc_z, lag_z, c=[palette_lisa[c] for c in gdf['lisa_cat']],
                s=70, alpha=0.85, edgecolors='white', linewidth=0.5)
poly = np.polyfit(ipc_z, lag_z, 1)
x_l = np.linspace(ipc_z.min(), ipc_z.max(), 100)
axes[1].plot(x_l, np.polyval(poly, x_l), 'k-', lw=1.5, alpha=0.7)
axes[1].axhline(0,color='gray',lw=0.5); axes[1].axvline(0,color='gray',lw=0.5)
axes[1].set_xlabel('IPC standardisé'); axes[1].set_ylabel('Lag IPC standardisé')
axes[1].set_title('Diagramme de Moran', fontweight='bold')
plt.tight_layout(); plt.show()

---
# §5 — Features de Distance et d'Accessibilité

## 5.1 Pourquoi l'enclavement prédit-il l'insécurité ?

> 🌍 **Trois effets de l'enclavement au Sahel :**
> 1. Accès limité aux marchés → impossibilité d'importer lors d'un déficit local
> 2. Aide humanitaire ralentie → les convois mettent plus de temps à arriver
> 3. Surveillance déficiente → signalement tardif des crises émergentes

## 5.2 Distance aux villes principales (KDTree)

Le **KDTree** est une structure de données qui permet de trouver le point
le plus proche en O(log n) au lieu de O(n). Indispensable pour les grandes couches.

> ⚠️ Toujours reprojeter en EPSG:32630 avant de calculer des distances.
> En degrés (4326), une distance de 1° vaut ~111 km à l'équateur mais moins aux hautes latitudes.

In [ ]:
from pyproj import Transformer

VILLES = {
    'Ouagadougou':    (12.3647, -1.5243),
    'Bobo-Dioulasso': (11.1771, -4.2978),
    'Koudougou':      (12.2525, -2.3683),
    'Banfora':        (10.6333, -4.7667),
    'Ouahigouya':     (13.5742, -2.4197),
    'Dori':           (14.0350,  0.0343),
}

# Transformer WGS84 → UTM 30N
tf = Transformer.from_crs(4326, 32630, always_xy=True)
gdf_utm = gdf.to_crs(32630)
prov_xy  = np.array([(g.centroid.x, g.centroid.y) for g in gdf_utm.geometry])
villes_xy = np.array([tf.transform(lon, lat) for lat, lon in VILLES.values()])
noms_v   = list(VILLES.keys())

# KDTree : recherche du voisin le plus proche en O(log n)
tree = cKDTree(villes_xy)
dists, idx = tree.query(prov_xy, k=1)
gdf['dist_ville_km'] = dists / 1e3
gdf['ville_proche']  = [noms_v[i] for i in idx]

# Corrélation distance → IPC
print('=== Corrélation distance ville → phase IPC ===')
corr = gdf[['dist_ville_km','ipc_phase']].corr().iloc[0,1]
print(f'Corrélation Pearson : {corr:.3f}')
print('→ Corrélation positive attendue : plus loin = plus en crise')

# Carte
fig, ax = plt.subplots(figsize=(10, 8))
gdf.plot(column='dist_ville_km', cmap='YlOrRd', legend=True,
         legend_kwds={'label':'Distance ville (km)','orientation':'vertical'},
         edgecolor='white', linewidth=0.5, ax=ax)
for nom, (lat, lon) in VILLES.items():
    ax.plot(lon, lat, 'b^', markersize=9, zorder=5)
    ax.annotate(nom[:4], xy=(lon, lat), xytext=(3,3),
                textcoords='offset points', fontsize=7.5, color='navy', fontweight='bold')
ax.set_title('Distance à la ville principale la plus proche (km)', fontweight='bold')
ax.axis('off'); plt.tight_layout(); plt.show()

## 5.3 Distance au réseau routier (HOT OSM)

La distance au réseau routier est plus fine que la distance aux villes :
elle capture l'**enclavement réel** même pour une province proche d'une grande ville
mais sans route praticable.

> 📂 Source : HOT / OpenStreetMap — Burkina Faso Roads (HDX, 600+ téléchargements)
> Format GeoJSON, ~50 Mo. Si indisponible : 6 axes routiers principaux simulés.

In [ ]:
# Chargement routes HOT OSM ou fallback axes principaux
URL_ROADS = ''  # URL HOT OSM si disponible — laisser vide pour fallback
roads_loaded = False

if URL_ROADS:
    try:
        r = requests.get(URL_ROADS, timeout=TIMEOUT); r.raise_for_status()
        roads_raw = gpd.read_file(io.BytesIO(r.content))
        roads_utm = roads_raw.to_crs(32630)
        roads_union = roads_utm.geometry.unary_union
        src_roads = 'HOT OSM (réel)'
        roads_loaded = True
    except: pass

if not roads_loaded:
    # Fallback : 6 axes routiers principaux BF
    segs = [
        [tf.transform(-1.5243,12.3647), tf.transform(-2.3683,12.2525)],  # Ouaga-Koudougou
        [tf.transform(-1.5243,12.3647), tf.transform(-4.2978,11.1771)],  # Ouaga-Bobo
        [tf.transform(-1.5243,12.3647), tf.transform(-2.4197,13.5742)],  # Ouaga-Ouahigouya
        [tf.transform(-1.5243,12.3647), tf.transform( 0.0343,14.0350)],  # Ouaga-Dori
        [tf.transform(-4.2978,11.1771), tf.transform(-4.7667,10.6333)],  # Bobo-Banfora
        [tf.transform(-2.4197,13.5742), tf.transform(-2.3683,12.2525)],  # Ouahigouya-Koudougou
    ]
    from shapely.geometry import MultiLineString
    roads_union = MultiLineString([LineString(s) for s in segs])
    src_roads = 'Axes principaux simulés (fallback)'

gdf['dist_route_km'] = gdf_utm.geometry.centroid.distance(roads_union) / 1e3
print(f'Source routes : {src_roads}')
print(f'dist_route_km : moy={gdf["dist_route_km"].mean():.1f} km |',
      f'max={gdf["dist_route_km"].max():.1f} km')

# Accessibilité marché
gdf['acces_marche'] = pd.cut(gdf['dist_route_km'],
                              bins=[0,80,200,9999],
                              labels=['Bon','Moyen','Difficile'])
print('\nAccessibilité aux marchés :')
print(gdf['acces_marche'].value_counts())

---
# §6 — Features Spectrales : FAPAR et Végétation

## 6.1 Pourquoi la végétation satellite prédit-elle l'insécurité ?

> 🌍 **Au Burkina Faso, 90% de la production agricole est pluviale.**
> La végétation — mesurée par satellite — est le **proxy le plus direct**
> de la production céréalière locale (mil, sorgho, niébé).

**FAPAR** (Fraction of Absorbed Photosynthetically Active Radiation) :
proportion du rayonnement solaire absorbé par les plantes pour la photosynthèse.

| Indice | Plage | Avantage pour le Sahel |
|--------|-------|------------------------|
| NDVI | [−1, 1] | Simple mais **sature** à haute densité végétale |
| **FAPAR** | **[0, 1]** | **Proportionnel à la photosynthèse réelle — non saturant** |

## 6.2 Génération du raster FAPAR (synthétique ou Copernicus)

> 📂 Source réelle : Copernicus FAPAR BF — HDX (100+ téléchargements, mensuel, ~2 Mo/fichier)
> Fallback : gradient N-S calibré sur les données Copernicus 2012–2024.

In [ ]:
import rasterio
from rasterio.transform import from_bounds
from rasterstats import zonal_stats
from scipy.stats import linregress

BBOX_BF = (-5.5, 9.4, 2.4, 15.1)
SHAPE   = (80, 100)

def make_fapar(shape, bbox, month=7, seed=1):
    np.random.seed(seed)
    rows, cols = shape
    lats = np.linspace(bbox[3], bbox[1], rows)
    lons = np.linspace(bbox[0], bbox[2], cols)
    LON, LAT = np.meshgrid(lons, lats)
    pluie = max(0.0, float(np.sin((month - 2) * np.pi / 6)))
    base  = (0.55 - (LAT - 9.4) * 0.04) * (0.15 + 0.85 * pluie)
    return np.clip(base + np.random.normal(0, 0.03, shape), 0, 1).astype(np.float32)

# Génération raster juillet (saison des pluies — végétation maximale)
fapar_july = make_fapar(SHAPE, BBOX_BF, month=7)

# Écriture GeoTIFF
transform = from_bounds(*BBOX_BF, width=SHAPE[1], height=SHAPE[0])
profile = {'driver':'GTiff','dtype':rasterio.float32,
           'width':SHAPE[1],'height':SHAPE[0],
           'count':1,'crs':'EPSG:4326','transform':transform,'nodata':-9999.0}
RASTER_PATH = 'fapar_cm_demo.tif'
with rasterio.open(RASTER_PATH, 'w', **profile) as dst:
    dst.write(fapar_july, 1)  # ← indices bandes commencent à 1

# Visualisation du raster
fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(fapar_july, cmap='RdYlGn', vmin=0, vmax=0.7,
               extent=[BBOX_BF[0],BBOX_BF[2],BBOX_BF[1],BBOX_BF[3]],
               origin='upper')
plt.colorbar(im, ax=ax, label='FAPAR [0–1]')
ax.set_title('FAPAR synthétique — juillet (gradient sahélien N-S)',
             fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout(); plt.show()

print(f'FAPAR juillet : min={fapar_july.min():.3f} | max={fapar_july.max():.3f}')
print('Observation : FAPAR faible au Nord (Sahel aride), élevé au Sud (Soudanien).')

## 6.3 Statistiques zonales — du raster continu à la feature tabulaire

> 🎯 **rasterstats** calcule des agrégats d'un raster dans chaque polygone.
> C'est la technique clé pour passer d'une donnée continue (raster pixel)
> à des features tabulaires par unité administrative.

Il gère automatiquement :
- L'alignement raster/vecteur (même CRS requis)
- Les valeurs nodata (pixels invalides exclus)
- Les polygones qui débordent le raster (recadrage automatique)

In [ ]:
# Statistiques zonales FAPAR par province
stats = zonal_stats(gdf, RASTER_PATH,
                    stats=['mean','std','min','max'], nodata=-9999.0)

gdf['fapar_mean'] = [s['mean'] for s in stats]
gdf['fapar_std']  = [s['std']  for s in stats]

col = 'province' if 'province' in gdf.columns else gdf.columns[0]
print('=== FAPAR moyen par province (5 extrêmes) ===')
print('5 plus faibles (Sahel) :')
print(gdf[[col,'fapar_mean','ipc_phase']].nsmallest(5,'fapar_mean').to_string(index=False))
print('\n5 plus élevés (Sud) :')
print(gdf[[col,'fapar_mean','ipc_phase']].nlargest(5,'fapar_mean').to_string(index=False))

# Corrélation FAPAR → IPC (doit être négative : + végétation → - insécurité)
corr_fapar = gdf[['fapar_mean','ipc_phase']].corr().iloc[0,1]
print(f'\nCorrélation fapar_mean / ipc_phase : {corr_fapar:.3f}')
print('→ Corrélation négative attendue : provinces avec peu de végétation → IPC élevé')

## 6.4 Tendance FAPAR — lire la trajectoire des terres

> 🎯 **Deux features, deux horizons temporels :**
> - `fapar_mean` → production agricole de la **campagne en cours**
> - `fapar_trend` → trajectoire à **moyen terme** (dégradation ou verdissement)

> 🌍 **Signal terrain** : une province avec fapar_trend < −0.005/mois sur 5 ans
> est en désertification progressive — risque structurel d'insécurité future.
> C'est un signal **précurseur** que le Cadre Harmonisé ne capture pas encore.

In [ ]:
# Série temporelle 12 mois + calcul de la tendance
months = np.arange(1, 13)
n_prov = len(gdf)
fapar_series = np.zeros((n_prov, 12))

for m in months:
    arr_m = make_fapar(SHAPE, BBOX_BF, month=int(m), seed=int(m))
    tmp = f'fapar_cm_m{m:02d}.tif'
    with rasterio.open(tmp, 'w', **profile) as dst:
        dst.write(arr_m, 1)
    st = zonal_stats(gdf, tmp, stats=['mean'], nodata=-9999.0)
    fapar_series[:, m-1] = [s['mean'] or 0 for s in st]

# Régression linéaire mois → FAPAR pour chaque province
gdf['fapar_trend'] = [linregress(months, fapar_series[i]).slope for i in range(n_prov)]

# Visualisation : profils saisonniers de 4 provinces contrastées
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Profils saisonniers
exemples_idx = [0, 5, 10, 18]  # Sahel, Nord, Centre, Sud-Ouest
colors_ex = ['#e74c3c','#e67e22','#f1c40f','#27ae60']
for j, (idx, color) in enumerate(zip(exemples_idx, colors_ex)):
    if idx < len(gdf):
        nom = str(gdf.iloc[idx][col])[:10]
        axes[0].plot(months, fapar_series[idx], '-o', color=color,
                     label=f'{nom} (IPC {int(gdf.iloc[idx]["ipc_phase"])})', linewidth=2)
axes[0].set_xlabel('Mois'); axes[0].set_ylabel('FAPAR moyen')
axes[0].set_title('Profils saisonniers FAPAR\n4 provinces contrastées', fontweight='bold')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Carte tendance
gdf.plot(column='fapar_trend', cmap='RdYlGn', legend=True,
         legend_kwds={'label':'Tendance FAPAR (unités/mois)','orientation':'vertical'},
         edgecolor='white', linewidth=0.5, ax=axes[1])
axes[1].set_title('Tendance FAPAR (rouge=dégradation · vert=verdissement)', fontweight='bold')
axes[1].axis('off')
plt.tight_layout(); plt.show()

---
# §7 — Sélection de Features et Feature Matrix Finale

## 7.1 Pourquoi éliminer la multicolinéarité ?

Deux features très corrélées apportent **la même information deux fois**.
Pour un Random Forest, cela biaise les importances de features sans améliorer les prédictions.

> **VIF (Facteur d'Inflation de la Variance)** : mesure à quel point
> une feature peut être prédite par les autres.

| VIF | Interprétation | Action |
|-----|----------------|--------|
| < 5 | Acceptable | Conserver |
| 5–10 | Modérée | Surveiller |
| **> 10** | **Sévère** | **Éliminer** |

> 🌍 **Exemple BF** : `dist_ouaga_km` et `dist_ville_km` sont très corrélées
> car Ouagadougou est souvent la ville la plus proche. → Garder une seule.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

FEATURES = [
    'superficie_km2','pop_density','compacite',
    'fapar_mean','fapar_std','fapar_trend',
    'ipc_lag',
    'dist_route_km','dist_ville_km',
]

feats_ok = [f for f in FEATURES if f in gdf.columns]
X_raw = gdf[feats_ok + ['ipc_phase']].dropna()

# Normalisation AVANT VIF (obligatoire)
X_norm = MinMaxScaler().fit_transform(X_raw[feats_ok])

vif_df = pd.DataFrame({
    'Feature': feats_ok,
    'VIF': [variance_inflation_factor(X_norm, i) for i in range(X_norm.shape[1])]
}).sort_values('VIF', ascending=False)

print('=== Facteurs VIF ===')
print(vif_df.to_string(index=False))

haut_vif = vif_df[vif_df['VIF'] > 10]['Feature'].tolist()
if haut_vif:
    print(f'\n⚠️ À exclure (VIF > 10) : {haut_vif}')
else:
    print('\n✅ Aucun VIF > 10')

## 7.2 Feature matrix finale et export

La feature matrix est le **livrable central** de ce module.
Elle sera l'entrée directe du Module 3 pour l'entraînement des modèles ML.

| Ligne | Colonne | Contenu |
|-------|---------|---------|
| Province | Feature | Valeur normalisée [0,1] |
| + | ipc_phase | Variable cible (1–5) |

In [ ]:
feats_finales = [f for f in feats_ok if f not in haut_vif]

X_fin = MinMaxScaler().fit_transform(X_raw[feats_finales])
feature_matrix = pd.DataFrame(X_fin, columns=feats_finales)
feature_matrix['ipc_phase'] = X_raw['ipc_phase'].values
col = 'province' if 'province' in gdf.columns else gdf.columns[0]
feature_matrix['province'] = gdf.loc[X_raw.index, col].values

# Export CSV
feature_matrix.to_csv('M2_feature_matrix_BF.csv', index=False)
print('✅ Feature matrix exportée : M2_feature_matrix_BF.csv')
print(f'   {feature_matrix.shape[0]} provinces × {len(feats_finales)} features + 1 target')
print(f'   Features retenues : {feats_finales}')

# Heatmap de corrélation
fig, ax = plt.subplots(figsize=(max(8,len(feats_finales)), max(6,len(feats_finales)-1)))
corr = pd.DataFrame(X_fin, columns=feats_finales).corr()
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Corrélation')
lbls = [f.replace('_',' ') for f in feats_finales]
ax.set_xticks(range(len(feats_finales))); ax.set_xticklabels(lbls, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(feats_finales))); ax.set_yticklabels(lbls, fontsize=9)
ax.set_title(f'Matrice de corrélation — Feature Matrix M2\n({feature_matrix.shape[0]} provinces)',
             fontweight='bold')
for i in range(len(feats_finales)):
    for j in range(len(feats_finales)):
        val = corr.iloc[i,j]
        ax.text(j,i,f'{val:.2f}',ha='center',va='center',fontsize=8,
                color='black' if abs(val)<0.7 else 'white')
plt.tight_layout(); plt.show()

---
# §8 — Synthèse du Module 2

## Ce que vous avez appris et pourquoi ça compte

| Compétence | Technique | Impact sur la prédiction ML |
|-----------|-----------|----------------------------|
| Charger GADM + fallback | `load_gadm()` | Géométries précises → features fiables |
| CRS et reprojection | `to_crs(32630)` | Superficies/distances physiquement correctes |
| Features géométriques | `.area`, `.length`, formule compacité | Proxy de la logistique humanitaire |
| Matrice W Queen | `Queen.from_dataframe()` | Encode le voisinage réel des provinces |
| Lag spatial IPC | `lag_spatial(W, ipc)` | Capte la contagion régionale des crises |
| Test de Moran I | `esda.moran.Moran` | Justifie la spatial CV en Module 3 |
| LISA locaux | `Moran_Local` | Identifie hotspots et coldspots → feature catégorielle |
| Distance routes | `gpd.distance(routes)` | Proxy de l'enclavement physique |
| FAPAR zonal | `rasterstats.zonal_stats` | Du raster satellite à la feature tabulaire |
| Tendance FAPAR | `scipy.stats.linregress` | Signal précurseur de dégradation à long terme |
| VIF | `variance_inflation_factor` | Évite que la redondance biaise les importances RF |

## Vers le Module 3

> La feature matrix exportée (`M2_feature_matrix_BF.csv`) est l'entrée directe du Module 3.
> Les étapes suivantes seront :
> 1. **Random Forest et XGBoost** sur cette feature matrix
> 2. **Spatial cross-validation** (justifiée par le Moran I calculé ici)
> 3. **Interprétabilité** (SHAP values) — quelles features prédisent le mieux les phases IPC ?
> 4. **Prédiction cartographique** — carte des phases IPC prédites pour toutes les provinces

> 🎯 **Résultat attendu en M3** : `ipc_lag` devrait être dans le top-3 des features
> les plus importantes, ce qui validera rétrospectivement tout le travail de construction de W.